# Lezione 46 — Coppie chosen/rejected

> **Modulo:** Feedback e preference training · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 45 (schema feedback).
>
> Obiettivo pratico unico: trasformare feedback grezzo in **coppie
> (chosen, rejected)** pulite, la struttura dati che il reward model (Lezione
> 47) e DPO (Lezione 48) consumano.

## Teoria essenziale

RLHF e DPO non imparano da voti isolati ma da **coppie**: per un dato contesto,
una risposta *scelta* (chosen, $y_w$) e una *rifiutata* (rejected, $y_l$). Da
dove vengono?

- direttamente da feedback `preference` (Lezione 45);
- oppure **derivate**: da voti assoluti (una memoria upvotata vs una downvotata
  nello stesso contesto), o da un segnale esistente (es. importanza).

Regole di igiene dei dati: niente coppie con chosen == rejected, niente
duplicati, e — importante — bilanciare per non insegnare scorciatoie (es. "il
testo piu' lungo vince").

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

train = pd.read_csv(Path('..') / 'datasets' / 'processed' / 'memory_train.csv')
pref = train[train['type'] == 'preference'].reset_index(drop=True)
print(f"memorie di tipo preference: {len(pref)}")
print(pref[['memory_id', 'text', 'importance']].head(4).to_string(index=False))

memorie di tipo preference: 61
memory_id                                                          text  importance
hist_0000                              The user likes walking meetings.        0.68
hist_0001 The user prefers morning sessions for il progetto TensorFlow.        0.43
hist_0005                              The user likes walking meetings.        0.53
hist_0006                         The user dislikes late notifications.        0.21


In [2]:
rng = np.random.default_rng(46)

# derivo coppie da un segnale esistente: a parita' di contesto, la memoria con
# importanza piu' alta e' 'chosen', quella con importanza piu' bassa 'rejected'.
def costruisci_coppie(df, n_coppie=40, margine=0.15):
    coppie = []
    idx = df.index.to_numpy()
    tentativi = 0
    while len(coppie) < n_coppie and tentativi < 2000:
        tentativi += 1
        a, b = rng.choice(idx, size=2, replace=False)
        ia, ib = df.loc[a, 'importance'], df.loc[b, 'importance']
        if abs(ia - ib) < margine:          # troppo simili: segnale ambiguo, salto
            continue
        if df.loc[a, 'text'] == df.loc[b, 'text']:   # stesso testo: coppia degenere, salto
            continue
        chosen, rejected = (a, b) if ia > ib else (b, a)
        coppie.append({"chosen": df.loc[chosen, 'text'],
                       "rejected": df.loc[rejected, 'text'],
                       "imp_chosen": df.loc[chosen, 'importance'],
                       "imp_rejected": df.loc[rejected, 'importance']})
    return pd.DataFrame(coppie)

coppie = costruisci_coppie(pref)
print(f"coppie costruite: {len(coppie)}")
print(coppie.head(3).to_string(index=False))

coppie costruite: 40
                                                   chosen                                             rejected  imp_chosen  imp_rejected
                    The user dislikes late notifications.    The user prefers short updates about la palestra.        0.52          0.31
The user prefers morning sessions for il corso di cucina. The user prefers morning sessions for la newsletter.        0.70          0.34
                    The user dislikes late notifications.                     The user likes walking meetings.        0.60          0.39


Leggi le coppie: per costruzione la `chosen` ha importanza maggiore della
`rejected`, con un margine minimo che scarta i casi ambigui. Ora verifichiamo
di non aver introdotto una scorciatoia banale come "la piu' lunga vince".

In [3]:
len_chosen = coppie['chosen'].str.len().mean()
len_rejected = coppie['rejected'].str.len().mean()
print(f"lunghezza media chosen: {len_chosen:.1f} | rejected: {len_rejected:.1f}")

frazione_chosen_piu_lunga = (coppie['chosen'].str.len()
                             > coppie['rejected'].str.len()).mean()
print(f"frazione di coppie in cui la chosen e' piu' lunga: {frazione_chosen_piu_lunga:.2f}")
print("(vicino a 0.5 = nessuna scorciatoia di lunghezza; lontano = bias da correggere)")

lunghezza media chosen: 47.1 | rejected: 45.2
frazione di coppie in cui la chosen e' piu' lunga: 0.60
(vicino a 0.5 = nessuna scorciatoia di lunghezza; lontano = bias da correggere)


## Il progetto: Memory AI Lab, passo 46

Impacchetto la costruzione delle coppie con le regole d'igiene (niente
chosen==rejected, margine minimo) come funzione riusabile che il reward model
consumera'.

In [4]:
def coppie_valide(df):
    assert (df['imp_chosen'] > df['imp_rejected']).all()      # per costruzione
    assert (df['chosen'] != df['rejected']).all()             # niente coppie degeneri
    return len(df)

n = coppie_valide(coppie)
assert n > 0
print(f"coppie valide pronte per il reward model: {n}")

coppie valide pronte per il reward model: 40


## Riepilogo (max 8 punti)

1. RLHF/DPO imparano da **coppie** (chosen $y_w$, rejected $y_l$), non da voti
   isolati.
2. Le coppie vengono da feedback `preference` o sono **derivate** da altri
   segnali.
3. Qui derivate dall'importanza: piu' alta = chosen.
4. Un **margine minimo** scarta le coppie ambigue.
5. Niente chosen == rejected, niente duplicati.
6. **Attenzione alle scorciatoie**: es. "la piu' lunga vince".
7. Si controlla il bias di lunghezza (frazione vicino a 0.5).
8. Le coppie pulite alimentano reward model (47) e DPO (48).

## Quiz

1. Perche' scartare le coppie con importanza quasi uguale?
2. Cosa segnala una frazione "chosen piu' lunga" molto lontana da 0.5?
3. Perche' le coppie sono preferibili ai voti assoluti per il tuning?

*(Risposte: 1. il segnale di preferenza sarebbe ambiguo/rumoroso; 2. un bias di
lunghezza che il modello potrebbe sfruttare come scorciatoia; 3. codificano un
confronto diretto, piu' coerente e informativo per l'ottimizzazione.)*